In [ ]:
from luxai_s3.params import EnvParams

from luxai_s3.wrappers import LuxAIS3GymEnv
import numpy as np
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, CheckpointCallback
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize, SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback
from typing import Callable
import torch
from wrappers import SB3LuxEnvBase

In [ ]:
def make_env(seed=None, player_id='player_0', opponent_strategy='random'):
    """
    Create and wrap a Lux S3 environment for SBX/Stable Baselines 3.
    
    Args:
        seed: Random seed for reproducibility
        player_id: ID of the player to train ('player_0' or 'player_1')
        opponent_strategy: Strategy for the opponent ('random', 'static', etc.)
    """
    # Create the base environment
    env = LuxAIS3GymEnv()
    
    # Create environment parameters
    env_params = EnvParams(map_type=0, max_steps_in_match=100)
    
    # Apply our wrapper with explicit player_id and opponent strategy
    wrapped_env = SB3LuxEnvBase(env, player_id=player_id, opponent_strategy=opponent_strategy)
    
    # Reset to initialize observation and action spaces
    seed_val = seed if seed is not None else np.random.randint(10000)
    wrapped_env.reset(seed=seed_val, options=dict(params=env_params))
    
    return wrapped_env

## Check if the environment is compatible with SB3

In [ ]:
# Create the environment
env = make_env(seed=42)  # Using a fixed seed for reproducibility

# Print observation space shape
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

check_env(env)
print("Environment check passed successfully!")

In [ ]:
# episodes = 2

# for episode in range(episodes):
#     obs, info = env.reset()
#     done = False
#     rewards = 0
#     while not done:
#         action = env.action_space.sample()
#         obs, reward, done, trncated, info = env.step(action)
#         rewards += reward
#         env.render()
#         if done or trncated:
#             done = True
#     print("🏁 Episode finished 🏁", rewards)
# env.close()

In [ ]:
class EnhancedTensorboardCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(EnhancedTensorboardCallback, self).__init__(verbose)
        self.match_rewards = [
            [] for _ in range(5)
        ]  # For tracking rewards across 5 matches
        self.episode_rewards = []
        self.episode_lengths = []
        self.episode_count = 0

    def _on_step(self):
        for info in self.locals.get("infos", []):
            metrics = info["lux_metrics"]
            if "episode" in info:
                # Basic episode metrics
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_lengths.append(info["episode"]["l"])
                self.episode_count += 1

                # Log to TensorBoard
                self.logger.record(
                    "rollout/ep_rew_mean",
                    sum(self.episode_rewards[-100:]) / len(self.episode_rewards[-100:]),
                )
                self.logger.record(
                    "rollout/ep_len_mean",
                    sum(self.episode_lengths[-100:]) / len(self.episode_lengths[-100:]),
                )

                # You can log individual episode rewards too
                self.logger.record(f"episode/reward", info["episode"]["r"])
                self.logger.record(f"episode/length", info["episode"]["l"])
                if "match_number" in metrics and "game_number" in metrics:
                    match_num = metrics["match_number"]
                    game_num = metrics["game_number"]
                    self.logger.record(
                        f"lux/game_{game_num}_match_{match_num}_reward",
                        info["episode"]["r"],
                    )

                    # Track reward progression across matches
                    if 0 <= match_num < 5:
                        self.match_rewards[match_num].append(info["episode"]["r"])
                        match_avg = sum(self.match_rewards[match_num]) / len(
                            self.match_rewards[match_num]
                        )
                        self.logger.record(
                            f"lux/match_{match_num}_avg_reward", match_avg
                        )
                        
            # print(f"Match {metrics['match_number']}, Game {metrics['game_number']}, Reward: {info['episode']['r']}")
            # Resource metrics
            if "energy_collected" in metrics:
                value = metrics["energy_collected"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/energy_collected", float_value)
            if "total_energy" in metrics:
                value = metrics["total_energy"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/total_energy", float_value)
            if "collection_efficiency" in metrics:
                value = metrics["collection_efficiency"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/collection_efficiency", float_value)

            # Exploration metrics
            if "map_coverage" in metrics:
                value = metrics["map_coverage"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/map_coverage", float_value)
            if "new_tiles_revealed" in metrics:
                value = metrics["new_tiles_revealed"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/new_tiles_revealed", float_value)

            # Pathfinding metrics
            if "path_completion_rate" in metrics:
                value = metrics["path_completion_rate"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/path_completion_rate", float_value)
            if "optimal_path_adherence" in metrics:
                value = metrics["optimal_path_adherence"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/optimal_path_adherence", float_value)

            if "points_earned" in metrics:
                value = metrics["points_earned"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/points_earned", float_value)
            if "rule_reward" in metrics:
                value = metrics["rule_reward"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/rule_reward", float_value)
            if "sap_reward" in metrics:
                value = metrics["sap_reward"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/sap_reward", float_value)
            if "sap_actions_taken" in metrics:
                value = metrics["sap_actions_taken"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/sap_actions_taken", float_value)
            if "point_reward" in metrics:
                value = metrics["point_reward"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/point_reward", float_value)
            if "final_reward" in metrics:
                value = metrics["final_reward"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/final_reward", float_value)

            if "relic_control_streak" in metrics:
                value = metrics["relic_control_streak"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/relic_control_streak", float_value)
            if "visited_tiles_count" in metrics:
                value = metrics["visited_tiles_count"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/visited_tiles_count", float_value)
            if "relic_point_tiles_found" in metrics:
                value = metrics["relic_point_tiles_found"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/relic_point_tiles_found", float_value)
            # Win metrics
            if "win_rate" in metrics:
                value = metrics["win_rate"]
                # Convert JAX array to standard float
                if hasattr(value, "item"):
                    float_value = float(value.item())
                else:
                    float_value = float(value)
                self.logger.record("lux/win_rate", float_value)
            
            if "unit_positions" in metrics:
                unit_positions = metrics["unit_positions"]
                # Log each unit's position
                for idx, pos in enumerate(unit_positions):
                    self.logger.record(f"lux/unit_{idx}_position", pos)

        return True

In [ ]:
log_dir = "./ppo_lux_logs_base/"
models_dir = "./ppo_lux_model_base/"
SAVE_FREQUENCY = 1000
os.makedirs(log_dir, exist_ok=True)

envs = SubprocVecEnv([lambda: make_env(seed=i) for i in range(8)])
envs = VecNormalize(envs, norm_obs=True, norm_reward=True)

# Create callbacks
enchared_tb_callback = EnhancedTensorboardCallback()
checkpoint_callback = CheckpointCallback(
    save_freq=SAVE_FREQUENCY, save_path=models_dir, name_prefix="ppo_lux_model_base"
)

# Combine callbacks
callbacks = CallbackList([enchared_tb_callback, checkpoint_callback])

def linear_schedule(initial_value: float, final_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate
    :param final_value: Final learning rate
    :return: schedule that computes current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining: 1.0 - (current_timestep / total_timesteps)
        :return: current learning rate
        """
        return final_value + progress_remaining * (initial_value - final_value)

    return func

mps_device = None
if not torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print("MPS not available because the current PyTorch install was not "
              "built with MPS enabled.")
    else:
        print("MPS not available because the current MacOS version is not 12.3+ "
              "and/or you do not have an MPS-enabled device on this machine.")
else:
    mps_device = torch.device("mps")

    print("MPS available and enabled.")

# Create PPO model
model = PPO(
    "MultiInputPolicy",
    envs,
    verbose=1,
    learning_rate=linear_schedule(5e-4, 1e-5),
    n_steps=2048,        
    batch_size=128, 
    n_epochs=3,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    vf_coef=0.4,
    ent_coef=0.3,
    clip_range_vf=0.2,
    tensorboard_log="./ppo_lux_tensorboard/",
    device=mps_device,
    policy_kwargs=dict(
        net_arch=[256, 256], 
        activation_fn=torch.nn.ReLU
    )
)

# Train the model with callbacks
model.learn(total_timesteps=2000000, callback=callbacks)

# Save final model
model.save("ppo_lux_model_base_2")